# Part 2 - Ứng dụng Data Fitting vào dữ liệu thực tế

In [ ]:
from pathlib import Path
import json
import math
import pickle
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == "part2":
    ROOT = ROOT.parent

for path in (ROOT, ROOT / "part2", ROOT / "part1"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from config import RANDOM_STATE
from data_pipeline import (
    DataPipeline,
    load_dataset,
    train_test_split_frame,
    select_model_columns,
    write_eda_outputs,
    write_preprocessing_diagnostic_plots,
)
from model_comparison import run_model_comparison
from advanced_methods import run_kernel_ridge_bonus
from residual_analysis import residual_plots

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", font_scale=0.9)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

DATA_PATH = ROOT / "part2" / "data" / "data.csv"
OUT_DIR = ROOT / "part2" / "output"
EDA_DIR = OUT_DIR / "eda"
DIAG_DIR = OUT_DIR / "diagnostics"
TARGET = "pl_rade"
ALPHA = 0.05

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Root: {ROOT}")
print(f"Data: {DATA_PATH}")
print(f"Output: {OUT_DIR}")
print(f"random_state = {RANDOM_STATE}, alpha = {ALPHA}")


Cell setup import các module đã viết sẵn, cố định `random_state = 42` theo `config.py` và dùng mức ý nghĩa thống kê `alpha = 0.05`. Target của bài toán là `pl_rade`, tức bán kính hành tinh theo đơn vị bán kính Trái Đất.


## 1. Giới thiệu bài toán

Bài toán đặt ra là: từ các thông tin quan sát về quỹ đạo, transit, khối lượng hành tinh và đặc trưng sao chủ, ta có thể dự báo bán kính của ngoại hành tinh hay không?

Trong pipeline, target `pl_rade` được biến đổi thành `log_pl_rade = log(1 + pl_rade)`. Vì vậy các chỉ số RMSE, MAE và R2 chính được đọc trên thang log, còn phần Prediction sẽ biến đổi ngược về thang gốc để tính MAPE dễ diễn giải hơn.


## 2. Hiểu dữ liệu (Data Understanding)

### 2.1 Mô tả dữ liệu


In [ ]:
source_raw = load_dataset(DATA_PATH)
planet_raw = select_model_columns(source_raw, target=TARGET)

print("Shape dữ liệu chuẩn full-column:", source_raw.shape)
print("Shape sau schema filter:", planet_raw.shape)
display(planet_raw.head())

variable_table = pd.DataFrame(
    [
        ("pl_orbper", "Chu kỳ quỹ đạo", "ngày"),
        ("pl_orbsmax", "Bán trục lớn quỹ đạo", "AU"),
        ("pl_orbeccen", "Độ lệch tâm quỹ đạo", "không đơn vị"),
        ("pl_trandur", "Thời lượng transit", "giờ"),
        ("pl_trandep", "Độ sâu transit", "tỷ lệ/đơn vị dữ liệu gốc"),
        ("pl_imppar", "Impact parameter", "không đơn vị"),
        ("pl_eqt", "Nhiệt độ cân bằng hành tinh", "K"),
        ("pl_insol", "Thông lượng bức xạ nhận được", "so với Trái Đất"),
        ("pl_bmasse", "Khối lượng hành tinh", "khối lượng Trái Đất"),
        ("pl_rade", "Bán kính hành tinh - biến mục tiêu", "bán kính Trái Đất"),
        ("st_teff", "Nhiệt độ hiệu dụng sao chủ", "K"),
        ("st_rad", "Bán kính sao chủ", "bán kính Mặt Trời"),
        ("st_mass", "Khối lượng sao chủ", "khối lượng Mặt Trời"),
        ("st_met", "Kim loại tính của sao chủ", "dex"),
        ("st_logg", "Log-g của sao chủ", "log10(cm/s^2)"),
        ("sy_dist", "Khoảng cách tới hệ sao", "parsec"),
    ],
    columns=["Biến", "Ý nghĩa", "Đơn vị/ghi chú"],
)
display(variable_table)


`data.csv` là dữ liệu chuẩn của Part 2 với 4636 dòng và 320 cột gốc từ NASA Exoplanet Archive. Notebook và pipeline không đưa toàn bộ 320 cột vào mô hình; bước schema filter giữ 16 biến vật lý có ý nghĩa trực tiếp cho bài toán dự báo `pl_rade`, gồm nhóm quỹ đạo, transit, năng lượng nhận từ sao chủ, khối lượng hành tinh, đặc trưng sao chủ và khoảng cách hệ sao.


### Động lực vật lý 
Cấu trúc nên một hành tinh khí khổng lồ (Gas Giant) khác đáng kể so với một hành tinh đá (Terrestrial) hoặc Super-Earth. Nếu giữ nguyên toàn bộ tập dữ liệu, mặt phẳng hồi quy sẽ phải đồng thời mô tả nhiều chế độ vật lý khác nhau, dễ bị nhiễu bởi hiện tượng phương sai không đồng nhất do chênh lệch quy mô quá lớn giữa các nhóm hành tinh. Vì vậy, nhóm tối ưu hóa khả năng nội suy cục bộ của mô hình bằng cách chỉ giữ nhóm hành tinh đá và Super-Earth theo ngưỡng phân rã Fulton (Fulton Gap): bán kính không vượt quá 1.6R⊕ và khối lượng không vượt quá 10M⊕. Thao tác này được thực hiện trong pipeline ngay sau schema filter và trước train/test split, MICE, VIF cũng như chuẩn hóa, để bộ nội suy và các tham số mean/std chỉ học phân phối của riêng nhóm hành tinh này.


In [ ]:
planet = planet_raw[(planet_raw["pl_rade"] <= 1.6) & (planet_raw["pl_bmasse"] <= 10)].dropna(subset=[TARGET]).reset_index(drop=True)
print("Shape sau domain restriction:", planet.shape)
display(
    pd.DataFrame(
        {
            "Giai đoạn": ["data.csv", "Sau schema filter", "Sau domain restriction"],
            "Số dòng": [len(source_raw), len(planet_raw), len(planet)],
            "Số cột": [source_raw.shape[1], planet_raw.shape[1], planet.shape[1]],
        }
    )
)


Notebook giữ nhóm hành tinh đá/Super-Earth với `pl_rade <= 1.6` và `pl_bmasse <= 10`.


### 2.2 Khám phá dữ liệu (EDA)


In [ ]:
eda_paths = write_eda_outputs(planet, target=TARGET, output_dir=EDA_DIR)
describe_table = pd.read_csv(EDA_DIR / "describe_numeric.csv", index_col=0)
display(describe_table.round(4))


Bảng thống kê mô tả cho thấy dữ liệu có nhiều thang đo khác nhau. Các biến như `pl_orbper`, `pl_insol`, `sy_dist` có khoảng giá trị rộng và đuôi phải dài; target `pl_rade` đã bị giới hạn trong miền hành tinh đá/Super-Earth nên dao động hẹp hơn.


In [ ]:
display(Image(filename=str(EDA_DIR / "histograms.png")))


Histogram xác nhận nhiều biến bị lệch phải mạnh, đặc biệt là các biến quỹ đạo/năng lượng và khoảng cách hệ sao. Đây là lý do pipeline dùng log-transform cho các biến có thang đo dương và phân phối lệch.


In [ ]:
display(Image(filename=str(EDA_DIR / "correlation_heatmap.png")))


Heatmap tương quan cho thấy một số nhóm biến có quan hệ tuyến tính mạnh, ví dụ nhóm quỹ đạo/năng lượng. Vì các biến tương quan cao có thể làm hệ số OLS kém ổn định, pipeline cần bước lọc đa cộng tuyến bằng VIF trước khi fit mô hình cuối.


In [ ]:
display(Image(filename=str(EDA_DIR / "scatter_top5_target.png")))


Các scatter plot giữa target và những biến tương quan cao nhất cho thấy có tín hiệu dự báo rõ, nhưng quan hệ không hoàn toàn tuyến tính và còn nhiều điểm nhiễu. Điều này giải thích vì sao mô hình tuyến tính đạt R2 tốt nhưng vẫn chưa giải thích toàn bộ biến thiên của bán kính hành tinh.


### 2.3 Kiểm tra chất lượng dữ liệu


In [ ]:
duplicate_count = int(planet.duplicated().sum())
missing_table = pd.read_csv(EDA_DIR / "missing_values.csv")

print(f"Số dòng duplicate: {duplicate_count}/{len(planet)}")
display(missing_table[missing_table["missing_count"] > 0].head(12))

numeric_cols = planet.select_dtypes(include="number").columns.tolist()
fig, axes = plt.subplots(4, 4, figsize=(16, 10))
axes = axes.ravel()
for ax, col in zip(axes, numeric_cols):
    ax.boxplot(planet[col].dropna(), vert=True, patch_artist=True)
    ax.set_title(col)
    ax.set_ylabel("Value")
for ax in axes[len(numeric_cols):]:
    ax.set_visible(False)
fig.suptitle("Boxplot trước xử lý outlier", y=1.02)
fig.tight_layout()
plt.show()


Không có dòng trùng lặp sau domain restriction. Missing values xuất hiện ở nhiều biến, nhiều nhất là `pl_orbeccen`, `pl_insol`, `pl_orbsmax` và `pl_eqt`, nên xóa dòng sẽ làm mất dữ liệu và có thể gây lệch mẫu. Boxplot cũng cho thấy nhiều giá trị cực trị; trong bối cảnh thiên văn, các điểm này có thể là quan sát thật nên nhóm xây dựng pipeline giảm ảnh hưởng bằng log-transform/Winsorization thay vì xóa cứng.


## 3. Chuẩn bị dữ liệu (Data Preparation)

### 3.1 Lựa chọn biến

Từ dữ liệu NASA gốc, nhóm giữ các biến có ý nghĩa dự báo trực tiếp cho bán kính hành tinh: nhóm quỹ đạo, transit, năng lượng, khối lượng hành tinh, đặc trưng sao chủ và khoảng cách hệ sao. Các cột metadata, sai số đo lường, cờ giới hạn và định danh không được đưa vào mô hình vì không đại diện cho đặc tính vật lý độc lập của hành tinh.

Trong pipeline, `st_mass` và `st_logg` được loại sớm để giảm trùng lặp thông tin với các biến sao chủ khác; sau đó VIF tiếp tục loại các biến đa cộng tuyến còn lại.


### 3.2 Pipeline tiền xử lý


In [ ]:
train_raw, test_raw = train_test_split_frame(planet, test_size=0.2, random_state=RANDOM_STATE)

pipeline = DataPipeline(target=TARGET)
X_train, y_train = pipeline.fit_transform(train_raw)
X_test, y_test = pipeline.transform(test_raw, include_target=True)

preprocessed = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train.reset_index(drop=True),
    "y_test": y_test.reset_index(drop=True),
    "feature_names": pipeline.feature_names_,
    "target": TARGET,
    "target_transformed": pipeline.target_transformed_,
    "metadata": pipeline.metadata(),
    "eda_paths": eda_paths,
}
preprocessed["metadata"].update(
    {
        "source_data_path": str(DATA_PATH),
        "source_data_shape": list(source_raw.shape),
        "model_input_shape_before_domain_filter": list(planet_raw.shape),
        "schema_filter_columns": list(planet_raw.columns),
        "schema_filter_dropped_columns": int(source_raw.shape[1] - planet_raw.shape[1]),
    }
)

with (OUT_DIR / "preprocessed.pkl").open("wb") as f:
    pickle.dump(preprocessed, f)
(OUT_DIR / "preprocessing_metadata.json").write_text(json.dumps(preprocessed["metadata"], indent=2), encoding="utf-8")
if pipeline.missing_report_ is not None:
    pipeline.missing_report_.to_csv(OUT_DIR / "missing_report_after_transforms.csv", index=False)

prep_plot_paths = write_preprocessing_diagnostic_plots(train_raw, pipeline, DIAG_DIR)

print("Train/Test raw:", train_raw.shape, test_raw.shape)
print("Train/Test sau pipeline:", X_train.shape, X_test.shape)
print("Target transform:", TARGET, "->", pipeline.target_transformed_)
print("Final features:", pipeline.feature_names_)
print("VIF dropped columns:", pipeline.vif_drop_columns_)


Pipeline được fit trên train set và chỉ transform test set bằng các tham số học từ train để tránh data leakage. Các bước chính gồm log-transform target/predictors, Winsorization, MICE imputation, lọc VIF và chuẩn hóa feature.


In [ ]:
display(pd.DataFrame(pipeline.log_transform_report_).round(4))
display(Image(filename=str(DIAG_DIR / "log_transform_before_after.png")))


Log-transform làm giảm độ lệch và nén các biến có đuôi phải dài. Với target, mô hình học `log(1 + pl_rade)`, giúp phần dư ổn định hơn so với hồi quy trực tiếp trên thang gốc.


In [ ]:
display(pd.DataFrame(pipeline.winsor_report_).round(4))
display(Image(filename=str(DIAG_DIR / "winsorization_before_after.png")))


Winsorization giới hạn các giá trị quá cực đoan theo ngưỡng học từ train set. Cách này giữ lại quan sát nhưng giảm ảnh hưởng đòn bẩy của outlier lên hệ số hồi quy.


In [ ]:
imputation_report = pd.DataFrame(pipeline.imputation_report_)
display(imputation_report.round(4))
if (DIAG_DIR / "mice_observed_vs_imputed.png").exists():
    display(Image(filename=str(DIAG_DIR / "mice_observed_vs_imputed.png")))
else:
    print("Không có hình MICE vì tập train sau các bước trước không còn cột cần impute.")


MICE được dùng để điền missing values dựa trên quan hệ giữa các biến thay vì thay thế toàn bộ bằng một hằng số đơn giản. Bảng so sánh observed/imputed giúp kiểm tra giá trị điền vào có cùng miền phân phối hợp lý với dữ liệu quan sát hay không.


In [ ]:
vif_history = pd.DataFrame(pipeline.vif_history_)
final_vif = pd.DataFrame(
    {"feature": list(pipeline.final_vif_.keys()), "VIF": list(pipeline.final_vif_.values())}
).sort_values("VIF", ascending=False)
print("VIF history:")
display(vif_history)
print("Final VIF:")
display(final_vif.round(4))
if (DIAG_DIR / "vif_before_after.png").exists():
    display(Image(filename=str(DIAG_DIR / "vif_before_after.png")))


VIF loại các biến có đa cộng tuyến cao trước khi mô hình hóa. Sau bước này, ma trận feature còn lại ổn định hơn cho OLS và cũng giúp diễn giải hệ số Ridge/Lasso rõ hơn.


## 4. Xây dựng mô hình (Modeling)

### 4.1 OLS full


In [ ]:
model_result = run_model_comparison(OUT_DIR / "preprocessed.pkl", OUT_DIR, include_lasso=True, verbose=True)

ols_inference = pd.read_csv(OUT_DIR / "ols_inference.csv")
display(ols_inference.round(6))


Ở mức ý nghĩa `alpha = 0.05`, các biến có ý nghĩa thống kê trong OLS full là những biến có `p_value < 0.05`. Trong kết quả hiện tại, nhóm biến nổi bật gồm thời lượng/độ sâu transit, impact parameter, khối lượng hành tinh sau log-transform và `log_sy_dist`.


### 4.2 OLS selected


In [ ]:
with (OUT_DIR / "results.pkl").open("rb") as f:
    fitted = pickle.load(f)

selected_features = fitted["selected_features"]
selected_table = ols_inference[ols_inference["feature"].isin(["intercept"] + selected_features)].reset_index(drop=True)
print("Selected features:", selected_features)
display(selected_table.round(6))


OLS selected giữ lại các biến có bằng chứng thống kê mạnh hơn theo p-value. Mô hình rất gần OLS full nhưng hy sinh một phần rất nhỏ độ phù hợp để dễ diễn giải hơn và giảm nhiễu từ các biến ít ý nghĩa.


### 4.3 Ridge - chọn lambda


In [ ]:
ridge_cv = pd.read_csv(OUT_DIR / "ridge_lambda_search.csv").sort_values("lambda")
ridge_best = ridge_cv.loc[ridge_cv["mean_cv_mse"].idxmin()]

display(ridge_cv.sort_values("mean_cv_mse").round(6))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(ridge_cv["lambda"], ridge_cv["mean_cv_mse"], marker="o")
ax.axvline(ridge_best["lambda"], color="red", linestyle="--", label=f"lambda* = {ridge_best['lambda']:.4g}")
ax.set_xscale("log")
ax.set_title("Ridge CV-MSE theo lambda")
ax.set_xlabel("lambda")
ax.set_ylabel("Mean CV MSE")
ax.legend()
plt.show()


Ridge chọn lambda bằng cross-validation trên train set. Đường CV-MSE khá phẳng quanh vùng tốt nhất, cho thấy regularization vừa phải giúp ổn định hệ số nhưng không làm thay đổi quá mạnh chất lượng dự báo.


### 4.4 Lasso - chọn lambda


In [ ]:
lasso_cv = pd.read_csv(OUT_DIR / "lasso_lambda_search.csv").sort_values("lambda")
lasso_best = lasso_cv.loc[lasso_cv["mean_cv_mse"].idxmin()]

display(lasso_cv.sort_values("mean_cv_mse").round(6))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(lasso_cv["lambda"], lasso_cv["mean_cv_mse"], marker="o")
ax.axvline(lasso_best["lambda"], color="red", linestyle="--", label=f"lambda* = {lasso_best['lambda']:.4g}")
ax.set_xscale("log")
ax.set_title("Lasso CV-MSE theo lambda")
ax.set_xlabel("lambda")
ax.set_ylabel("Mean CV MSE")
ax.legend()
plt.show()


Lasso cũng được chọn lambda bằng 5-fold cross-validation trên train set. Vì Lasso có cơ chế kéo một số hệ số về gần 0, mô hình này vừa regularize vừa thực hiện chọn biến mềm, nên thường dễ cân bằng giữa dự báo và diễn giải.


## 5. Dự báo (Prediction)


In [ ]:
summary = fitted["summary"].copy().sort_values("test_R2", ascending=False).reset_index(drop=True)
best_name = str(summary.loc[0, "model"])
best_result = fitted["results"][best_name]

y_test_log = np.asarray(fitted["y_test"], dtype=float)
y_pred_log = np.asarray(best_result["test_pred"], dtype=float)
residual_log = y_test_log - y_pred_log

pred_table = pd.DataFrame(
    {
        "actual_log": y_test_log,
        "predicted_log": y_pred_log,
        "residual_log": residual_log,
        "actual_pl_rade": np.expm1(y_test_log),
        "predicted_pl_rade": np.expm1(y_pred_log),
    }
)
pred_table["abs_pct_error"] = (pred_table["actual_pl_rade"] - pred_table["predicted_pl_rade"]).abs() / pred_table["actual_pl_rade"].replace(0, np.nan) * 100
mape = pred_table["abs_pct_error"].mean()

print("Best model:", best_name)
display(pred_table.head(10).round(6))
print(f"MAPE trên thang gốc: {mape:.3f}%")


Bảng dự báo hiển thị 10 quan sát đầu tiên của test set. Sai số chính được mô hình tối ưu trên thang log, còn MAPE sau khi biến đổi ngược về thang gốc giúp diễn giải trực tiếp theo phần trăm sai lệch bán kính hành tinh.


In [ ]:
display(Image(filename=str(OUT_DIR / "actual_vs_predicted.png")))


Actual vs Predicted của mô hình tốt nhất nằm khá sát đường chéo, nghĩa là dự báo nhìn chung bám tốt giá trị thực. Các điểm lệch khỏi đường chéo là phần mà mô hình tuyến tính/log-linear chưa giải thích được hoàn toàn.


## 6. Đánh giá mô hình (Evaluation)

### 6.1 So sánh 4 mô hình


In [ ]:
display(summary[["model", "test_MAE", "test_RMSE", "test_R2", "cv_MSE", "cv_R2"]].round(6))
display(Image(filename=str(OUT_DIR / "model_comparison.png")))


Lasso đang nhỉnh nhất theo R2 test, nhưng OLS full và Ridge rất gần nhau. Điều này cho thấy sau khi pipeline đã xử lý skewness, missing values, outlier và VIF, tín hiệu còn lại tương đối tuyến tính; regularization chỉ cải thiện nhẹ chứ không tạo khác biệt lớn.


### 6.2 Feature importance


In [ ]:
display(Image(filename=str(OUT_DIR / "feature_importance.png")))

ridge = fitted["results"]["Ridge"]
ridge_coef = pd.DataFrame(
    {
        "feature": ridge["feature_names"],
        "coef": ridge["model"]["beta_hat"][1:],
    }
)
ridge_coef["abs_coef"] = ridge_coef["coef"].abs()
display(ridge_coef.sort_values("abs_coef", ascending=False).drop(columns="abs_coef").round(6))


Hệ số Ridge được đọc trên feature đã chuẩn hóa, nên độ lớn tuyệt đối phản ánh mức ảnh hưởng tương đối. `log_pl_bmasse` thường là biến mạnh nhất vì khối lượng liên quan trực tiếp tới kích thước hành tinh. `log_sy_dist` cũng quan trọng nhưng cần diễn giải cẩn thận: khoảng cách tới hệ sao có thể phản ánh selection bias của dữ liệu quan sát, vì các hệ xa/gần không được phát hiện và đo lường theo cùng một cơ chế hoàn toàn ngẫu nhiên.


### 6.3 Phân tích phần dư


In [ ]:
best_features = best_result["feature_names"]
X_test_best = fitted["X_test"][best_features].astype(float).values.tolist()

residual_info = residual_plots(
    y_test_log.tolist(),
    y_pred_log.tolist(),
    X=X_test_best,
    save_dir=str(OUT_DIR),
    show_plot=False,
)
display(Image(filename=residual_info["out_path"]))

residuals = residual_info["residuals"]
std_residuals = residual_info["std_residuals"]
cooks_d = residual_info["cooks_d"]
cook_threshold = 4 / len(cooks_d)

def _mean(values):
    return sum(values) / len(values) if values else 0.0

def _std(values):
    avg = _mean(values)
    return math.sqrt(sum((value - avg) ** 2 for value in values) / len(values)) if values else 0.0

diagnostic_summary = pd.DataFrame(
    [
        {
            "metric": "residual_mean",
            "value": _mean(residuals),
            "note": "Mean residual should be near 0",
        },
        {
            "metric": "residual_std",
            "value": _std(residuals),
            "note": "Residual spread on log target scale",
        },
        {
            "metric": "max_abs_standardized_residual",
            "value": max(abs(value) for value in std_residuals),
            "note": "Large values mark tail/outlier behavior",
        },
        {
            "metric": "cook_threshold_4_over_n",
            "value": cook_threshold,
            "note": "Reference line used in Cook distance plot",
        },
        {
            "metric": "cook_points_above_threshold",
            "value": sum(value > cook_threshold for value in cooks_d),
            "note": "Potentially influential observations",
        },
    ]
)
display(diagnostic_summary.round(6))


Bản diagnostic plots cho thấy mô hình tuyến tính mô tả được xu hướng trung tâm nhưng các giả định Gauss-Markov không hoàn toàn lý tưởng.

* **Residuals vs Fitted:** Phần lớn phần dư dao động quanh đường $e = 0$, nên giả định trung bình có điều kiện bằng 0 (GM1) nhìn chung chấp nhận được. Tuy nhiên, đường LOWESS không hoàn toàn nằm ngang: ở vùng fitted khoảng 0.82 - 0.90 phần dư có xu hướng dương, còn ở vùng fitted cao hơn lại giảm mạnh. Điều này gợi ý mô hình còn thiếu một phần cấu trúc phi tuyến hoặc tương tác.
* **Normal Q-Q:** Các điểm ở vùng giữa bám khá sát đường chuẩn, nhưng hai đuôi lệch rõ, đặc biệt đuôi trái có vài residual âm rất lớn. Vì vậy, giả định chuẩn của phần dư (GM5, nếu dùng cho suy luận mẫu nhỏ) chỉ đúng xấp xỉ; p-value Shapiro-Wilk nhỏ cũng ủng hộ nhận xét này.
* **Scale-Location:** Đường LOWESS không phẳng vì độ lớn phần dư tăng lại ở vùng fitted cao. Điều này là dấu hiệu phương sai phần dư chưa hoàn toàn đồng nhất, tức GM4 (homoskedasticity) bị vi phạm nhẹ đến vừa. Kết quả Breusch-Pagan có p-value nhỏ cũng gợi ý heteroskedasticity.
* **Cook's Distance:** Có 17 điểm vượt ngưỡng $4/n$, trong đó một điểm có Cook's distance rất lớn so với phần còn lại. Các điểm này có ảnh hưởng đáng kể đến mô hình, nhưng nhóm không xóa vì đây có thể là quan sát thiên văn hợp lý sau domain restriction, không phải lỗi nhập liệu rõ ràng.

### 6.4 Kiểm chứng giả thuyết Giãn nở nhiệt (Thermal Expansion Hypothesis)

**Kiểm định tác động tương tác giữa Khối lượng và Nhiệt độ:**

Từ đường cong trên biểu đồ Residuals vs Fitted, nhóm đặt giả thuyết rằng tác động của khối lượng lên bán kính có thể bị chi phối bởi mức độ bức xạ nhiệt, tức một dạng hiệu ứng giãn nở nhiệt.

Thực nghiệm đầu tiên bổ sung biến tương tác

$$\text{Khối lượng} \times \text{Nhiệt độ sao chủ}
= \texttt{log\_pl\_bmasse} \times \texttt{st\_teff}.$$

Kết quả cho thấy p-value của biến tương tác bằng $0.098 > 0.05$, nên biến này không có ý nghĩa thống kê ở mức ý nghĩa $\alpha = 0.05$. Nguyên nhân hợp lý là nhiệt độ sao chủ không phản ánh đúng nhiệt lượng hành tinh thực nhận nếu bỏ qua khoảng cách quỹ đạo.

Khi thay nhiệt độ sao chủ bằng bức xạ thực nhận, tức bổ sung biến tương tác

$$\text{Khối lượng} \times \text{Bức xạ thực nhận}
= \texttt{log\_pl\_bmasse} \times \texttt{log\_pl\_insol},$$

biến tương tác có ý nghĩa rất mạnh trên tập huấn luyện với p-value xấp xỉ 0. Tuy nhiên, hiệu năng ngoại suy lại suy giảm: $R^2_{\text{Test}}$ giảm từ khoảng $0.747$ xuống khoảng $0.741$. Điều này cho thấy biến tương tác tạo ra đa cộng tuyến cấu trúc (structural multicollinearity) và làm mô hình quá khớp với train set.

Kết luận rút ra là việc cố gắng bẻ cong mặt phẳng OLS bằng biến tương tác để sửa lỗi phần dư cục bộ đã phá vỡ tính tổng quát hóa của mô hình. Điều này củng cố nguyên lý Occam's Razor: mô hình OLS cơ bản hiện tại, không chứa biến tương tác, là điểm cân bằng hợp lý hơn giữa phương sai (variance) và độ chệch (bias).


### 6.5 Kernel Ridge (bonus)


In [ ]:
advanced = run_kernel_ridge_bonus(OUT_DIR / "preprocessed.pkl", OUT_DIR, max_train=200)
display(pd.DataFrame([advanced["best_on_test"]]).round(6))

kernel_search_path = OUT_DIR / "kernel_ridge_search.csv"
if kernel_search_path.exists():
    display(pd.read_csv(kernel_search_path).sort_values("Val_RMSE").head(10).round(6))


Kernel Ridge dùng RBF kernel như phần bonus để kiểm tra tín hiệu phi tuyến. Kết quả chỉ dùng tối đa 200 mẫu train do ma trận kernel tăng kích thước nhanh, nên không so sánh trực tiếp như một mô hình chính thức với bốn mô hình tuyến tính ở trên.


## 7. Kết luận

Pipeline tiền xử lý giúp đưa dữ liệu ngoại hành tinh về dạng phù hợp cho hồi quy: giảm lệch phân phối bằng log-transform, giới hạn outlier bằng Winsorization, điền missing values bằng MICE, xử lý đa cộng tuyến bằng VIF và chuẩn hóa feature.

Trong bốn mô hình chính, Lasso nhỉnh nhất trên test set nhưng OLS full và Ridge gần như tương đương. Điều này cho thấy phần lớn tín hiệu sau tiền xử lý có thể được mô tả khá tốt bằng quan hệ tuyến tính/log-linear. Tuy nhiên R2 test khoảng 0.75 nghĩa là vẫn còn khoảng 25% biến thiên chưa được giải thích.

Hạn chế chính là dữ liệu quan sát có selection bias, missing values và khả năng phi tuyến vật lý chưa được mô hình tuyến tính nắm bắt đầy đủ. Hướng cải thiện hợp lý là bổ sung biến vật lý/quan sát khác, kiểm tra mô hình phi tuyến một cách có kiểm soát và đánh giá độ bất định của dự báo trên thang gốc.
